# 사후설문 ANOVA 가정 검정과 Box–Cox 변환 재분석

이 노트북은 교수님이 제시한 분석 절차를 종속변수별로 적용합니다.

1. 원자료로 UI × 숙련도 이원분산분석을 실행합니다.
2. ANOVA 모형의 잔차로 정규성과 등분산성을 검정합니다.
3. 두 가정이 충족되면 원 ANOVA의 F값과 p값을 사용합니다.
4. 하나라도 충족되지 않으면 종속변수에 Box–Cox 변환을 적용합니다.
5. 변환 자료로 ANOVA와 잔차 가정 검정을 다시 실행합니다.
6. 변환 후 가정이 충족되면 변환 후 F값과 p값을 사용합니다.
7. 변환 후에도 가정이 충족되지 않으면 비모수 검정 대상으로 표시합니다.

각 단계는 설명, 짧은 코드, 결과 확인, 해석 순서로 구성했습니다.


별표 기준: ** p<.05, * .05<p<.10. p=.05와 p=.10에는 별표를 붙이지 않으며, 반올림 전 p값으로 판정합니다.


## 0. 판단 기준

| 단계 | 판단 |
|---|---|
| 잔차 정규성 | Shapiro–Wilk p≥.05이면 위반 근거 없음 |
| 잔차 등분산성 | 중앙값 기준 Levene p≥.05이면 위반 근거 없음 |
| 원자료에서 두 가정 충족 | 원 ANOVA 결과 채택 |
| 원자료에서 한 가정 이상 위반 | Box–Cox 변환 후 재분석 |
| 변환 후 두 가정 충족 | Box–Cox ANOVA 결과 채택 |
| 변환 후에도 한 가정 이상 위반 | 비모수 검정 필요 |

p≥.05는 가정이 완벽하게 성립한다는 증명이 아니라, 현재 자료에서 위반을 확인할 충분한 근거가 없다는 뜻입니다.


## 1. 분석 환경과 원자료 준비

### 1-1. 라이브러리 불러오기

Type III ANOVA에는 statsmodels, 가정 검정과 Box–Cox 변환에는 scipy를 사용합니다.


In [ ]:
from pathlib import Path
import sys

A2UI_DIR = Path.cwd() if (Path.cwd() / "analysis_common.py").exists() else Path.cwd() / "Data" / "A2UI"
if str(A2UI_DIR) not in sys.path:
    sys.path.insert(0, str(A2UI_DIR))
import analysis_common as common
from functools import partial
from analysis_common import significance_stars, format_p

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from IPython.display import Image, display


### 1-2. 입력 파일과 그림 저장 위치


In [ ]:
A2UI_DIR = common.A2UI_DIR
SURVEY_FILE = common.SURVEY_FILE
FIGURE_DIR = common.FIGURE_DIR
KOREAN_FONT = common.configure_plots()


### 1-3. 사후설문 원자료 읽기

04번과 04-1번 노트북에서 사용한 `병합 데이터` 시트를 그대로 사용합니다.


In [ ]:
survey = common.load_survey()

print(f"자료 크기: {survey.shape[0]}명 × {survey.shape[1]}개 열")
display(survey[["참여자번호", "인터페이스"]].head())


## 2. 7개 사후설문 척도와 집단 변수 만들기

### 2-1. 04번과 동일한 척도 문항 정의


In [ ]:
DV_ITEMS = common.DV_ITEMS
LITERACY_ITEMS = common.LITERACY_ITEMS


### 2-2. 문항 번호에 맞는 엑셀 열 찾기


In [ ]:
find_columns = partial(common.find_columns, survey)
average_score = partial(common.average_score, survey)


### 2-4. 종속변수 점수 계산하기


In [ ]:
scores = common.make_scores(survey)

assert scores.notna().all().all()
assert (scores > 0).all().all(), "Box–Cox 적용을 위해 점수가 0보다 커야 합니다."
display(scores.head().round(2))


### 2-5. UI와 숙련도 분류하기

사전 AI 리터러시 평균 6.0 이상은 숙련자, 미만은 초보자로 분류합니다.


In [ ]:
design = common.make_design(survey)
literacy_pre = design["literacy_pre"]



In [ ]:
GROUP_ORDER = [
    "A2UI · 초보자", "A2UI · 숙련자",
    "Baseline · 초보자", "Baseline · 숙련자",
]
group_counts = design["집단"].value_counts().reindex(GROUP_ORDER)

assert group_counts.sum() == 40
display(group_counts.rename("n").to_frame())


---

## 3. 원자료 이원분산분석

### 3-1. 효과 코딩 Type III 모형 함수

집단별 표본 수는 각각 10명이며 UI, 숙련도, UI×숙련도를 동시에 통제하는 효과 코딩 기반 Type III ANOVA를 사용합니다.


In [ ]:
TERM_LABELS = common.TERM_LABELS
EFFECT_ORDER = list(TERM_LABELS.values())
fit_anova = partial(common.fit_anova, design)


### 3-2. ANOVA 표를 정리하는 함수

절편과 잔차 행을 제외하고 UI, 숙련도, UI×숙련도의 F값과 p값만 남깁니다.


In [ ]:
tidy_anova = common.tidy_anova


### 3-3. 7개 종속변수의 원 ANOVA 실행하기


In [ ]:
original_models = {}
original_model_data = {}
original_anova_tables = {}
original_anova_rows = []

for variable in DV_ITEMS:
    model, model_data, table = fit_anova(scores[variable])
    original_models[variable] = model
    original_model_data[variable] = model_data
    original_anova_tables[variable] = table
    original_anova_rows.append(
        tidy_anova(variable, "원자료", model, table)
    )

original_anova = pd.concat(original_anova_rows, ignore_index=True)


In [ ]:
original_anova_display = original_anova.copy()
original_anova_display["F"] = original_anova_display["F"].round(3)
original_anova_display["p"] = original_anova_display["p"].round(3)
display(original_anova_display.set_index(["종속변수", "효과"]))


## 4. 원 ANOVA 잔차의 가정 검정

### 4-1. 잔차 정규성과 등분산성을 함께 검사하는 함수

Shapiro–Wilk는 전체 모형 잔차에 적용합니다. Levene 검정은 같은 잔차를 UI×숙련도 네 집단으로 나누어 비교합니다.


In [ ]:
test_residual_assumptions = common.test_residual_assumptions


### 4-2. 원자료 모형의 잔차 가정 검정 실행하기


In [ ]:
original_assumption_rows = []
for variable in DV_ITEMS:
    result = test_residual_assumptions(
        original_models[variable],
        original_model_data[variable],
    )
    result["종속변수"] = variable
    result["단계"] = "원자료"
    original_assumption_rows.append(result)

original_assumptions = pd.DataFrame(original_assumption_rows)
original_assumptions["가정 충족"] = (
    (original_assumptions["정규성 p"] >= .05)
    & (original_assumptions["등분산성 p"] >= .05)
)


In [ ]:
original_assumption_display = original_assumptions.copy()
for column in ["Shapiro W", "정규성 p", "Levene F", "등분산성 p"]:
    original_assumption_display[column] = (
        original_assumption_display[column].round(3)
    )
original_assumption_display["판단"] = np.where(
    original_assumption_display["가정 충족"],
    "원 ANOVA 사용", "Box–Cox 필요",
)
display(original_assumption_display.set_index("종속변수"))


### 4-3. Box–Cox 대상 변수 확인하기

정규성 또는 등분산성 중 하나라도 p<.05인 종속변수만 변환합니다.


In [ ]:
boxcox_targets = original_assumptions.loc[
    ~original_assumptions["가정 충족"],
    "종속변수",
].tolist()

print(f"Box–Cox 대상 변수 수: {len(boxcox_targets)}개")
for variable in boxcox_targets:
    print(f"- {variable}")


**원자료 판단:** 위 결과표에서 정규성 또는 등분산성 p<.05인 변수를 자동으로 선택해 아래에서 Box–Cox 재분석합니다.


## 5. Box–Cox 변환과 ANOVA 재실행

### 5-1. Box–Cox 변환 함수

Box–Cox는 0보다 큰 값에만 적용할 수 있습니다. 현재 설문 점수는 모두 양수이므로 이동값은 0입니다. 함수는 재사용을 위해 0 이하 값이 있을 때만 최소한의 양수 이동을 적용하도록 작성합니다.


In [ ]:
def apply_boxcox(values):
    values = np.asarray(values, dtype=float)
    minimum = values.min()
    shift = 0.0 if minimum > 0 else 1.0 - minimum
    transformed, lambda_value = stats.boxcox(values + shift)
    return transformed, lambda_value, shift


### 5-2. 대상 변수 변환하기

λ는 scipy가 정규성에 가장 적합하도록 추정한 Box–Cox 변환계수입니다.


In [ ]:
boxcox_scores = {}
boxcox_metadata_rows = []

for variable in boxcox_targets:
    transformed, lambda_value, shift = apply_boxcox(
        scores[variable]
    )
    boxcox_scores[variable] = transformed
    boxcox_metadata_rows.append({
        "종속변수": variable,
        "Box–Cox λ": lambda_value,
        "양수 이동값": shift,
    })

boxcox_metadata = pd.DataFrame(boxcox_metadata_rows, columns=["종속변수", "Box–Cox λ", "양수 이동값"])
display(boxcox_metadata.set_index("종속변수").round(3))


### 5-3. 변환된 점수로 ANOVA 다시 실행하기


In [ ]:
boxcox_models = {}
boxcox_model_data = {}
boxcox_anova_rows = []

for variable in boxcox_targets:
    model, model_data, table = fit_anova(
        boxcox_scores[variable]
    )
    boxcox_models[variable] = model
    boxcox_model_data[variable] = model_data
    boxcox_anova_rows.append(
        tidy_anova(variable, "Box–Cox", model, table)
    )

boxcox_anova = pd.concat(boxcox_anova_rows, ignore_index=True) if boxcox_anova_rows else original_anova.iloc[:0].copy()


In [ ]:
boxcox_anova_display = boxcox_anova.copy()
boxcox_anova_display["F"] = boxcox_anova_display["F"].round(3)
boxcox_anova_display["p"] = boxcox_anova_display["p"].round(3)
display(boxcox_anova_display.set_index(["종속변수", "효과"]))


## 6. Box–Cox 변환 후 잔차 가정 재검정

### 6-1. 변환 모형의 Shapiro–Wilk와 Levene 검정


In [ ]:
boxcox_assumption_rows = []
for variable in boxcox_targets:
    result = test_residual_assumptions(
        boxcox_models[variable],
        boxcox_model_data[variable],
    )
    result["종속변수"] = variable
    result["단계"] = "Box–Cox"
    boxcox_assumption_rows.append(result)

boxcox_assumptions = pd.DataFrame(boxcox_assumption_rows, columns=["Shapiro W", "정규성 p", "Levene F", "등분산성 p", "종속변수", "단계"])
boxcox_assumptions["가정 충족"] = (
    (boxcox_assumptions["정규성 p"] >= .05)
    & (boxcox_assumptions["등분산성 p"] >= .05)
)


In [ ]:
boxcox_assumption_display = boxcox_assumptions.merge(
    boxcox_metadata, on="종속변수", how="left"
)
for column in [
    "Box–Cox λ", "Shapiro W", "정규성 p",
    "Levene F", "등분산성 p",
]:
    boxcox_assumption_display[column] = (
        boxcox_assumption_display[column].round(3)
    )
boxcox_assumption_display["판단"] = np.where(
    boxcox_assumption_display["가정 충족"],
    "Box–Cox ANOVA 사용", "비모수 검정 필요",
)
display(boxcox_assumption_display.set_index("종속변수"))


**변환 후 판단:** 위의 변환 후 가정 검정 결과를 기준으로 분석 경로를 결정합니다. 최종 채택 결과는 8절의 표를 확인합니다.


## 7. Box–Cox 전후 시각적 진단

### 7-1. Q–Q plot과 잔차 집단 분포 함수


In [ ]:
COLORS = {
    "original": "#D1495B",
    "boxcox": "#2A6F97",
    "line": "#59636E",
    "grid": "#D9DEE3",
}

def draw_qq(ax, residuals, title, color):
    theoretical, ordered = stats.probplot(
        residuals, dist="norm", fit=False
    )
    slope, intercept, _ = stats.probplot(
        residuals, dist="norm", fit=True
    )[1]
    ax.scatter(theoretical, ordered, s=28, color=color, alpha=.85)
    ax.plot(theoretical, slope * theoretical + intercept,
            color=COLORS["line"], linewidth=1.4)
    ax.set_title(title)
    ax.set_xlabel("이론적 분위수")
    ax.set_ylabel("잔차 분위수")


In [ ]:
def residual_frame(model, model_data):
    return pd.DataFrame({
        "집단": model_data["집단"],
        "잔차": model.resid.to_numpy(),
    })

def draw_residual_groups(ax, model, model_data, title):
    data = residual_frame(model, model_data)
    sns.boxplot(
        data=data, x="집단", y="잔차", hue="집단",
        order=GROUP_ORDER, palette="Set2",
        showfliers=False, legend=False, ax=ax,
    )
    sns.stripplot(
        data=data, x="집단", y="잔차", order=GROUP_ORDER,
        color="#263238", size=3.7, jitter=.15, alpha=.7, ax=ax,
    )
    ax.axhline(0, color=COLORS["line"], linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=24, labelsize=8)


### 7-2. 원자료와 변환 자료 비교 그림

위쪽은 원자료, 아래쪽은 Box–Cox 변환 결과입니다. 왼쪽은 점수 분포, 가운데는 잔차 Q–Q plot, 오른쪽은 네 집단의 잔차 분포입니다.


In [ ]:
diagnostic_paths = []
for number, variable in enumerate(boxcox_targets, start=1):
    path = FIGURE_DIR / f"figure_04_2_boxcox_diagnostic_{number}.png"
    with plt.rc_context({"font.family": KOREAN_FONT, "axes.unicode_minus": False}):
        fig, axes = plt.subplots(2, 3, figsize=(15, 8.4))
        sns.histplot(scores[variable], kde=True,
                     color=COLORS["original"], ax=axes[0, 0])
        axes[0, 0].set_title("원자료 점수 분포")
        draw_qq(axes[0, 1], original_models[variable].resid,
                "원자료 ANOVA 잔차 Q–Q", COLORS["original"])
        draw_residual_groups(axes[0, 2], original_models[variable],
                             original_model_data[variable], "원자료 집단별 잔차")
        sns.histplot(boxcox_scores[variable], kde=True,
                     color=COLORS["boxcox"], ax=axes[1, 0])
        axes[1, 0].set_title("Box–Cox 변환 점수 분포")
        draw_qq(axes[1, 1], boxcox_models[variable].resid,
                "변환 후 ANOVA 잔차 Q–Q", COLORS["boxcox"])
        draw_residual_groups(axes[1, 2], boxcox_models[variable],
                             boxcox_model_data[variable], "변환 후 집단별 잔차")
        fig.suptitle(f"{variable}: Box–Cox 변환 전후 진단", fontsize=15)
        fig.tight_layout()
        fig.savefig(path, dpi=300, bbox_inches="tight")
        plt.close(fig)
    diagnostic_paths.append(path)
    display(Image(filename=path))


### 7-3. 가정 검정 p값 전후 비교

점선은 유의수준 .05입니다. Box–Cox 변환 후 두 점이 모두 점선 오른쪽으로 이동하면 정규성과 등분산성 검정에서 위반 근거가 사라진 것입니다.


In [ ]:
before_after = pd.concat([
    original_assumptions.loc[
        original_assumptions["종속변수"].isin(boxcox_targets)
    ],
    boxcox_assumptions,
], ignore_index=True)
comparison_long = before_after.melt(
    id_vars=["종속변수", "단계"],
    value_vars=["정규성 p", "등분산성 p"],
    var_name="검정", value_name="p",
)


In [ ]:
comparison_png = FIGURE_DIR / "figure_04_2_assumption_before_after.png"

with plt.rc_context({"font.family": KOREAN_FONT, "axes.unicode_minus": False}):
    fig, ax = plt.subplots(figsize=(8.5, 5.4))
    sns.lineplot(
        data=comparison_long, x="단계", y="p", hue="종속변수",
        style="검정", markers=True, dashes=False,
        linewidth=2, markersize=9, ax=ax,
    )
    ax.axhline(.05, color="#C1121F", linestyle="--", linewidth=1.3)
    ax.set_ylim(bottom=0)
    ax.set_xlabel("")
    ax.set_ylabel("p값")
    ax.set_title("Box–Cox 변환 전후 잔차 가정 검정")
    ax.grid(axis="y", color=COLORS["grid"], linewidth=.7, alpha=.75)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(comparison_png, dpi=300, bbox_inches="tight")
    plt.close(fig)
display(Image(filename=comparison_png))


## 8. 종속변수별 최종 분석 방법 선택

### 8-1. 원 ANOVA, Box–Cox ANOVA, 비모수 검정 경로 결정


In [ ]:
route_rows = []
for variable in DV_ITEMS:
    original_ok = original_assumptions.loc[
        original_assumptions["종속변수"].eq(variable),
        "가정 충족",
    ].iloc[0]
    transformed_ok = (
        boxcox_assumptions.loc[
            boxcox_assumptions["종속변수"].eq(variable),
            "가정 충족",
        ].iloc[0]
        if variable in boxcox_targets else False
    )
    method = (
        "원 ANOVA" if original_ok
        else "Box–Cox ANOVA" if transformed_ok
        else "비모수 검정 필요"
    )
    route_rows.append({"종속변수": variable, "최종 분석 방법": method})
analysis_routes = pd.DataFrame(route_rows)
display(analysis_routes.set_index("종속변수"))


### 8-2. 최종 채택할 F값과 p값 만들기

가정이 처음부터 충족된 변수는 원 ANOVA, Box–Cox 후 충족된 변수는 변환 후 ANOVA 결과를 선택합니다.


In [ ]:
final_rows = []
for row in analysis_routes.itertuples(index=False):
    variable = row[0]
    method = row[1]
    if method == "원 ANOVA":
        selected = original_anova.loc[
            original_anova["종속변수"].eq(variable)
        ]
    elif method == "Box–Cox ANOVA":
        selected = boxcox_anova.loc[
            boxcox_anova["종속변수"].eq(variable)
        ]
    else:
        continue
    selected = selected.copy()
    selected["최종 분석 방법"] = method
    final_rows.append(selected)

final_anova = pd.concat(final_rows, ignore_index=True) if final_rows else original_anova.assign(**{"최종 분석 방법": ""}).iloc[:0].copy()


In [ ]:
final_anova_display = final_anova[
    ["종속변수", "최종 분석 방법", "효과", "F", "df1", "df2", "p"]
].copy()
final_anova_display["df"] = final_anova_display.apply(
    lambda row: f"{int(row['df1'])}, {int(row['df2'])}", axis=1
)
final_anova_display["F"] = final_anova_display["F"].round(3)
final_anova_display["p"] = final_anova_display["p"].map(
    lambda value: ("<.001" if value < .001 else f"{value:.3f}") + significance_stars(value)
)
display(final_anova_display[
    ["종속변수", "최종 분석 방법", "효과", "F", "df", "p"]
].set_index(["종속변수", "효과"]))


## 9. UI 유형별 평균 방향 비교

ANOVA의 p값만으로는 어느 UI의 평균이 더 높거나 낮은지 알 수 없으므로, UI 주효과를 해석할 네 변수의 A2UI와 Baseline 평균을 직접 비교합니다. p값은 8절에서 선택한 최종 ANOVA 결과를 사용하고, 평균은 해석이 쉬운 7점 원척도로 표시합니다.

> 인지부하(CL)는 낮을수록 긍정적이므로 다른 변수와 해석 방향이 반대입니다.


In [ ]:
comparison_variables = [
    ("전반적 만족도", "SAT", False),
    ("결과 신뢰도·품질", "PRC", False),
    ("재사용 의도", "RI", False),
    ("인지부하", "CL", True),
]

ui_means = (
    scores.assign(UI=design["UI"])
    .groupby("UI")[list(DV_ITEMS)]
    .mean()
)
ui_effects = (
    final_anova.loc[final_anova["효과"].eq("UI")]
    .set_index("종속변수")
)

comparison_rows = []
for variable, abbreviation, lower_is_better in comparison_variables:
    a2ui_mean = ui_means.loc["A2UI", variable]
    baseline_mean = ui_means.loc["Baseline", variable]
    comparison_rows.append({
        "종속변수": variable,
        "약어": abbreviation,
        "A2UI": a2ui_mean,
        "Baseline": baseline_mean,
        "부호": ">" if a2ui_mean > baseline_mean else "<",
        "p": ui_effects["p"].get(variable, np.nan),
        "최종 분석 방법": analysis_routes.set_index("종속변수").loc[variable, "최종 분석 방법"],
        "낮을수록 긍정적": lower_is_better,
    })

comparison_data = pd.DataFrame(comparison_rows)
comparison_display = comparison_data.copy()
comparison_display["평균 비교"] = comparison_display.apply(
    lambda row: (
        f"A2UI {row['A2UI']:.2f} {row['부호']} "
        f"{row['Baseline']:.2f} Baseline"
    ),
    axis=1,
)
comparison_display["p"] = comparison_display["p"].map(
    lambda value: "비모수 검정 필요" if pd.isna(value) else ("p<.001" if value < .001 else f"p={value:.3f}".replace("0.", ".")) + significance_stars(value)
)
display(comparison_display[["종속변수", "평균 비교", "p"]].set_index("종속변수"))


### 9-1. 논문용 평균 비교 그래프

막대는 A2UI와 Baseline 조건의 원척도 평균을 나타내며, 막대 위 숫자는 각 조건의 평균값입니다. 인지부하는 낮을수록 긍정적입니다.


In [ ]:
comparison_png = FIGURE_DIR / "figure_04_2_ui_mean_direction.png"
colors = {"A2UI": "#E4572E", "Baseline": "#4C78A8"}

with plt.rc_context({
    "font.family": KOREAN_FONT,
    "axes.unicode_minus": False,
    "font.size": 12,
}):
    fig, ax = plt.subplots(figsize=(12.8, 7.2))
    x_positions = np.arange(len(comparison_data))
    bar_width = .34
    a2ui_values = comparison_data["A2UI"].to_numpy()
    baseline_values = comparison_data["Baseline"].to_numpy()

    a2ui_bars = ax.bar(
        x_positions - bar_width / 2, a2ui_values, bar_width,
        label="A2UI", color=colors["A2UI"], zorder=3,
    )
    baseline_bars = ax.bar(
        x_positions + bar_width / 2, baseline_values, bar_width,
        label="Baseline", color=colors["Baseline"], zorder=3,
    )
    ax.bar_label(
        a2ui_bars, fmt="%.2f", padding=3,
        fontsize=12, fontweight="bold", color=colors["A2UI"],
    )
    ax.bar_label(
        baseline_bars, fmt="%.2f", padding=3,
        fontsize=12, color=colors["Baseline"],
    )

    x_labels = [
        (
            f"{row['종속변수']}\n({row['약어']})"
            + ("\n낮을수록 긍정적" if row["낮을수록 긍정적"] else "")
        )
        for _, row in comparison_data.iterrows()
    ]
    ax.set_xticks(x_positions, x_labels)
    ax.set_ylim(0, 7.25)
    ax.set_yticks(range(0, 8))
    ax.set_ylabel("7점 척도 평균", fontsize=12)
    ax.set_title(
        "A2UI와 Baseline의 주요 사후설문 변수 평균 비교",
        fontsize=18, fontweight="bold", pad=22,
    )
    ax.grid(axis="y", color="#E5E7EB", linewidth=.8, zorder=0)
    ax.legend(frameon=False, ncol=2, loc="upper right")
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.spines["left"].set_color("#D1D5DB")
    ax.spines["bottom"].set_color("#D1D5DB")
    fig.tight_layout()
    fig.savefig(comparison_png, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)

display(Image(filename=comparison_png))
print(f"저장 완료: {comparison_png}")


**그림. A2UI와 Baseline의 주요 사후설문 변수 평균 비교.** 막대는 각 UI의 원척도 평균입니다. 평균 방향은 기술통계이며, 통계적 판단은 8절의 최종 분석 방법과 결과를 기준으로 합니다. 인지부하는 낮을수록 긍정적입니다.


## 10. 인지부하(CL) Interaction Plot

각 점은 UI × AI 리터러시 집단의 원척도 평균입니다. 상호작용의 통계적 판단은 최신 ANOVA 결과표의 p값을 기준으로 합니다.


In [ ]:
cl_interaction_data = pd.concat(
    [design[["UI", "숙련도"]], scores[["인지부하"]]],
    axis=1,
)
cl_interaction_summary = (
    cl_interaction_data
    .groupby(["UI", "숙련도"], observed=True)["인지부하"]
    .agg(n="size", M="mean", SD="std")
    .reset_index()
)
cl_interaction_p = final_anova.loc[
    final_anova["종속변수"].eq("인지부하")
    & final_anova["효과"].eq("UI × 숙련도"),
    "p",
].iloc[0]

cl_interaction_display = cl_interaction_summary.copy()
cl_interaction_display[["M", "SD"]] = (
    cl_interaction_display[["M", "SD"]].round(2)
)
display(cl_interaction_display.set_index(["UI", "숙련도"]))
print(f"UI × AI 리터러시 상호작용: p={cl_interaction_p:.3f}".replace("0.", "."))


In [ ]:
cl_interaction_png = FIGURE_DIR / "figure_04_2_cl_interaction.png"
proficiency_order = ["초보자", "숙련자"]
ui_order = ["A2UI", "Baseline"]
interaction_styles = {
    "A2UI": {"marker": "o", "linestyle": "-"},
    "Baseline": {"marker": "^", "linestyle": (0, (2, 2))},
}

with plt.rc_context({
    "font.family": KOREAN_FONT,
    "axes.unicode_minus": False,
    "font.size": 12,
}):
    fig, ax = plt.subplots(figsize=(8.2, 5.9))
    x_positions = np.arange(len(proficiency_order))

    for ui in ui_order:
        ui_data = (
            cl_interaction_summary.loc[cl_interaction_summary["UI"].eq(ui)]
            .set_index("숙련도")
            .reindex(proficiency_order)
        )
        means = ui_data["M"].to_numpy()
        style = interaction_styles[ui]
        ax.plot(
            x_positions, means, label=ui,
            color="black", marker=style["marker"],
            linestyle=style["linestyle"], linewidth=2.5,
            markersize=9, zorder=3,
        )

        label_offset = 13 if ui == "A2UI" else -22
        vertical_alignment = "bottom" if ui == "A2UI" else "top"
        for x_value, mean_value in zip(x_positions, means):
            ax.annotate(
                f"{mean_value:.2f}",
                (x_value, mean_value),
                xytext=(0, label_offset), textcoords="offset points",
                ha="center", va=vertical_alignment,
                color="black", fontsize=11,
            )

    p_text = f"p={cl_interaction_p:.3f}{significance_stars(cl_interaction_p)}".replace("0.", ".")
    ax.text(
        .98, .96, f"UI × AI 리터러시: {p_text}",
        transform=ax.transAxes, ha="right", va="top", fontsize=11,
    )
    ax.set_xticks(x_positions, proficiency_order)
    ax.set_xlim(-.25, 1.25)
    ax.set_ylim(
        min(1.8, cl_interaction_summary["M"].min() - .35),
        max(4.0, cl_interaction_summary["M"].max() + .5),
    )
    ax.set_xlabel("AI 리터러시 집단", fontsize=12)
    ax.set_ylabel("인지부하 평균 (CL)", fontsize=12)
    ax.grid(True, color="#D9D9D9", linewidth=.8, alpha=.8, zorder=0)
    ax.legend(
        title="UI 유형", frameon=False,
        loc="center left", bbox_to_anchor=(1.02, .5),
    )
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#333333")
        spine.set_linewidth(1.0)
    fig.subplots_adjust(right=.78, bottom=.24, top=.93)
    fig.text(
        .42, .055, "인지부하 (CL)",
        ha="center", va="center", fontsize=15, fontweight="bold",
    )
    fig.savefig(
        cl_interaction_png, dpi=300, bbox_inches="tight", facecolor="white"
    )
    plt.close(fig)

display(Image(filename=cl_interaction_png))
print(f"저장 완료: {cl_interaction_png}")


**그림. UI 유형과 AI 리터러시에 따른 인지부하 평균.** 네 집단의 평균은 그림에 표시했습니다. 인지부하는 낮을수록 긍정적이며, UI × AI 리터러시 상호작용은 최신 ANOVA 결과표에서 확인합니다.


## 11. 종합표용 결과 저장

05번은 이 수치 결과를 읽습니다. 입력 파일과 공통 코드 해시가 다르면 이전 결과를 사용하지 않습니다. 비모수 검정이 필요한 변수는 F·p를 비워 표시합니다.

In [ ]:
import json
common.RESULT_DIR.mkdir(exist_ok=True)
complete_index = pd.MultiIndex.from_product([list(DV_ITEMS), EFFECT_ORDER], names=["종속변수", "효과"])
export_table = final_anova.set_index(["종속변수", "효과"]).reindex(complete_index).reset_index()
export_table["최종 분석 방법"] = export_table["종속변수"].map(analysis_routes.set_index("종속변수")["최종 분석 방법"])
export_table.to_csv(common.RESULT_DIR / "survey_final_anova.csv", index=False, encoding="utf-8-sig")
manifest = {"survey_sha256": common.survey_hash(), "common_sha256": common.module_hash(),
            "source_description": "Private participant survey; raw data not distributed"}
(common.RESULT_DIR / "survey_final_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("최종 결과 21행과 입력 검증 정보를 results/에 저장했습니다.")
